# CGRA Solutions — Chapter 4 (Ex 4.1–4.4)
GLM transforms. Open with **Voilà**.

In [ ]:
%%html
<script src="https://git.io/glm-js.min.js"></script>


In [ ]:
%%html
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecshader.js"></script>
<script src="https://is3l.isr.uc.pt/~pm/CGRA/JS/deecapp.js"></script>


In [ ]:
%%html
<script id="my-vertex-shader" type="x-shader/x-vertex">
precision highp float;
attribute vec2 in_Position;
attribute vec3 in_Color;
varying vec3 ex_Color;
void main(void){
  gl_Position = vec4(in_Position.x, in_Position.y, 0.0, 1.0);
  ex_Color = in_Color;
}
</script>
<script id="my-fragment-shader" type="x-shader/x-fragment">
precision highp float;
varying vec3 ex_Color;
void main(void){ gl_FragColor = vec4(ex_Color,1.0); }
</script>


## Ex 4.1 — small blue triangle follows sinusoid L→R, wraps

In [ ]:
%%html
<canvas id="c41" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>
class App41 extends DEECapp{
  initialize(){
    super.initialize();
    this.gl.clearColor(1,1,1,1);
    this.local = [
      glm.vec4(-0.05, 0.06, 0, 1),
      glm.vec4(-0.05,-0.06, 0, 1),
      glm.vec4( 0.05,-0.06, 0, 1)
    ];
    this.colors = [0,0,1, 0,0,1, 0,0,1];
    this.vertices = new Array(6);
    this.vbuf = this.gl.createBuffer();
    this.cbuf = this.gl.createBuffer();
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.cbuf);
    this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(this.colors), this.gl.STATIC_DRAW);
    this.t0 = Date.now();
  }
  render(){
    var t = (Date.now()-this.t0)/1000.0;
    var x = -1.0 + ((t*0.3) % 2.0); // wrap -1..1
    if(x > 1) x -= 2;
    var y = 0.4 * Math.sin(2.0 * Math.PI * x);
    var T = glm.mat4(1.0);
    T[3] = glm.vec4(x, y, 0, 1);
    for(var i=0;i<3;i++){
      var p = T['*'](this.local[i]);
      this.vertices[2*i]=p.x; this.vertices[2*i+1]=p.y;
    }
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vbuf);
    this.gl.bufferData(this.gl.ARRAY_BUFFER, new Float32Array(this.vertices), this.gl.DYNAMIC_DRAW);
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.vbuf);
    var lp=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Position');
    this.gl.vertexAttribPointer(lp,2,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lp);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER, this.cbuf);
    var lc=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Color');
    this.gl.vertexAttribPointer(lc,3,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lc);
    this.shaderprog.startUsing();
    this.gl.drawArrays(this.gl.TRIANGLES,0,3);
    this.shaderprog.stopUsing();
  }
}
new App41('c41').run();
</script>


## Ex 4.2 — same + rotate while moving

In [ ]:
%%html
<canvas id="c42" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>
class App42 extends DEECapp{
  initialize(){
    super.initialize();
    this.gl.clearColor(1,1,1,1);
    this.local = [glm.vec4(-0.05,0.06,0,1), glm.vec4(-0.05,-0.06,0,1), glm.vec4(0.05,-0.06,0,1)];
    this.colors = [0,0,1, 0,0.5,1, 0,1,1];
    this.vertices = new Array(6);
    this.vbuf = this.gl.createBuffer();
    this.cbuf = this.gl.createBuffer();
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.cbuf);
    this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.colors),this.gl.STATIC_DRAW);
    this.t0 = Date.now();
  }
  render(){
    var t=(Date.now()-this.t0)/1000.0;
    var x = -1.0 + ((t*0.3)%2.0); if(x>1)x-=2;
    var y = 0.4*Math.sin(2*Math.PI*x);
    var q = glm.angleAxis(t, glm.vec3(0,0,1));
    var R = glm.toMat4(q);
    var T = glm.mat4(1.0); T[3]=glm.vec4(x,y,0,1);
    var M = T['*'](R); // rotate then translate
    for(var i=0;i<3;i++){
      var p=M['*'](this.local[i]);
      this.vertices[2*i]=p.x; this.vertices[2*i+1]=p.y;
    }
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.vbuf);
    this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.vertices),this.gl.DYNAMIC_DRAW);
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.vbuf);
    var lp=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Position');
    this.gl.vertexAttribPointer(lp,2,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lp);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.cbuf);
    var lc=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Color');
    this.gl.vertexAttribPointer(lc,3,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lc);
    this.shaderprog.startUsing();
    this.gl.drawArrays(this.gl.TRIANGLES,0,3);
    this.shaderprog.stopUsing();
  }
}
new App42('c42').run();
</script>


## Ex 4.3 — rotating rectangle under moving triangle

In [ ]:
%%html
<canvas id="c43" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>
class App43 extends DEECapp{
  initialize(){
    super.initialize();
    this.gl.clearColor(1,1,1,1);
    this.triLocal=[glm.vec4(-0.05,0.06,0,1),glm.vec4(-0.05,-0.06,0,1),glm.vec4(0.05,-0.06,0,1)];
    this.rectLocal=[
      glm.vec4(-0.15,-0.08,0,1),glm.vec4(0.15,-0.08,0,1),glm.vec4(0.15,0.08,0,1),
      glm.vec4(-0.15,-0.08,0,1),glm.vec4(0.15,0.08,0,1),glm.vec4(-0.15,0.08,0,1)
    ];
    this.triC=[0,0,1,0,0,1,0,0,1];
    this.rectC=[1,0,0,1,0,0,1,0,0,1,0,0,1,0,0,1,0,0];
    this.triV=new Array(6); this.rectV=new Array(12);
    this.tv=this.gl.createBuffer(); this.rv=this.gl.createBuffer();
    this.tc=this.gl.createBuffer(); this.rc=this.gl.createBuffer();
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.tc);
    this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.triC),this.gl.STATIC_DRAW);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.rc);
    this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.rectC),this.gl.STATIC_DRAW);
    this.t0=Date.now();
  }
  draw(vbuf,cbuf,arr,n){
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,vbuf);
    this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(arr),this.gl.DYNAMIC_DRAW);
    var lp=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Position');
    this.gl.vertexAttribPointer(lp,2,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lp);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,cbuf);
    var lc=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Color');
    this.gl.vertexAttribPointer(lc,3,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lc);
    this.shaderprog.startUsing();
    this.gl.drawArrays(this.gl.TRIANGLES,0,n);
    this.shaderprog.stopUsing();
  }
  render(){
    var t=(Date.now()-this.t0)/1000.0;
    var x=-1+((t*0.3)%2); if(x>1)x-=2;
    var y=0.4*Math.sin(2*Math.PI*x);
    var qT=glm.angleAxis(t,glm.vec3(0,0,1));
    var RT=glm.toMat4(qT);
    var TT=glm.mat4(1.0); TT[3]=glm.vec4(x,y,0,1);
    var MT=TT['*'](RT);
    for(var i=0;i<3;i++){var p=MT['*'](this.triLocal[i]); this.triV[2*i]=p.x; this.triV[2*i+1]=p.y;}
    var qR=glm.angleAxis(-0.7*t,glm.vec3(0,0,1));
    var RR=glm.toMat4(qR);
    var TR=glm.mat4(1.0); TR[3]=glm.vec4(0,-0.55,0,1);
    var MR=TR['*'](RR);
    for(var i=0;i<6;i++){var p=MR['*'](this.rectLocal[i]); this.rectV[2*i]=p.x; this.rectV[2*i+1]=p.y;}
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    this.draw(this.rv,this.rc,this.rectV,6);
    this.draw(this.tv,this.tc,this.triV,3);
  }
}
new App43('c43').run();
</script>


## Ex 4.4 — clock (hour / minute / second hands as rectangles)

In [ ]:
%%html
<canvas id="c44" width="400" height="400" style="border:2px solid #000;"></canvas>
<script>
class App44 extends DEECapp{
  initialize(){
    super.initialize();
    this.gl.clearColor(1,1,1,1);
    // rectangle pointing UP (y+), pivot at origin bottom-center style: from (0,0) upward
    // hand local: thin rectangle along +Y
    this.makeHand = function(len, halfW){
      return [
        glm.vec4(-halfW, 0, 0, 1),
        glm.vec4( halfW, 0, 0, 1),
        glm.vec4( halfW, len, 0, 1),
        glm.vec4(-halfW, 0, 0, 1),
        glm.vec4( halfW, len, 0, 1),
        glm.vec4(-halfW, len, 0, 1)
      ];
    };
    this.hLocal = this.makeHand(0.35, 0.025);
    this.mLocal = this.makeHand(0.50, 0.018);
    this.sLocal = this.makeHand(0.62, 0.010);
    this.hC=[0.2,0.2,0.2, 0.2,0.2,0.2, 0.2,0.2,0.2, 0.2,0.2,0.2, 0.2,0.2,0.2, 0.2,0.2,0.2];
    this.mC=[0.1,0.1,0.6, 0.1,0.1,0.6, 0.1,0.1,0.6, 0.1,0.1,0.6, 0.1,0.1,0.6, 0.1,0.1,0.6];
    this.sC=[0.8,0,0, 0.8,0,0, 0.8,0,0, 0.8,0,0, 0.8,0,0, 0.8,0,0];
    // clock face circle approx as many triangles optional - skip, just hands
    this.hv=new Array(12); this.mv=new Array(12); this.sv=new Array(12);
    this.hb=this.gl.createBuffer(); this.mb=this.gl.createBuffer(); this.sb=this.gl.createBuffer();
    this.hc=this.gl.createBuffer(); this.mc=this.gl.createBuffer(); this.sc=this.gl.createBuffer();
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.hc); this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.hC),this.gl.STATIC_DRAW);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.mc); this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.mC),this.gl.STATIC_DRAW);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,this.sc); this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(this.sC),this.gl.STATIC_DRAW);
  }
  fill(local, ang, out){
    var q=glm.angleAxis(ang, glm.vec3(0,0,1));
    var R=glm.toMat4(q);
    for(var i=0;i<6;i++){
      var p=R['*'](local[i]);
      out[2*i]=p.x; out[2*i+1]=p.y;
    }
  }
  draw(vbuf,cbuf,arr){
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,vbuf);
    this.gl.bufferData(this.gl.ARRAY_BUFFER,new Float32Array(arr),this.gl.DYNAMIC_DRAW);
    var lp=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Position');
    this.gl.vertexAttribPointer(lp,2,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lp);
    this.gl.bindBuffer(this.gl.ARRAY_BUFFER,cbuf);
    var lc=this.gl.getAttribLocation(this.shaderprog.shaderProgram,'in_Color');
    this.gl.vertexAttribPointer(lc,3,this.gl.FLOAT,false,0,0);
    this.gl.enableVertexAttribArray(lc);
    this.shaderprog.startUsing();
    this.gl.drawArrays(this.gl.TRIANGLES,0,6);
    this.shaderprog.stopUsing();
  }
  render(){
    var today = new Date();
    var hours = today.getHours() % 12;
    var minutes = today.getMinutes();
    var secs = today.getSeconds();
    // angles: 12 o'clock = up = 0 in our local (+Y). Clockwise negative Z rotation in RH? 
    // Screen: clockwise for clock. angleAxis positive is CCW. So use negative.
    var angS = -secs * (2*Math.PI/60);
    var angM = -(minutes + secs/60) * (2*Math.PI/60);
    var angH = -(hours + minutes/60) * (2*Math.PI/12);
    this.fill(this.hLocal, angH, this.hv);
    this.fill(this.mLocal, angM, this.mv);
    this.fill(this.sLocal, angS, this.sv);
    this.gl.clear(this.gl.COLOR_BUFFER_BIT);
    this.draw(this.hb,this.hc,this.hv);
    this.draw(this.mb,this.mc,this.mv);
    this.draw(this.sb,this.sc,this.sv);
  }
}
new App44('c44').run();
</script>
